In [ ]:
"""
Script d'entraînement du modèle HR-Attrition-Predictor.
Ce script charge les données RH, nettoie et encode les variables,
entraîne un algorithme de Forêt Aléatoire (Random Forest) et sauvegarde
les artefacts (fichiers .pkl) nécessaires pour l'application web Streamlit.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib
import os

# ==========================================
# 1. PRÉPARATION DE L'ENVIRONNEMENT
# ==========================================
print("📂 Création du dossier models...")
# Crée le dossier 'models' s'il n'existe pas déjà pour stocker les fichiers sérialisés
os.makedirs('models', exist_ok=True)

# ==========================================
# 2. CHARGEMENT ET PRÉTRAITEMENT DES DONNÉES
# ==========================================
print("📊 Chargement des données RH...")
df = pd.read_csv('HR_Analytics_Data.csv')

# Séparation des caractéristiques/features (X) et de la variable cible à prédire (y)
X = df.drop('Attrition', axis=1)

# Transformation de la variable cible en valeurs binaires compréhensibles par la machine
# 'Oui' (démission) devient 1, 'Non' (reste) devient 0
y = df['Attrition'].map({'Oui': 1, 'Non': 0})

# ==========================================
# 3. ENCODAGE DES VARIABLES TEXTUELLES
# ==========================================
print("⚙️ Encodage des données textuelles...")
encoders = {}
# Identification automatique de toutes les colonnes contenant du texte (catégories)
categorical_cols = X.select_dtypes(include=['object']).columns

# Boucle sur chaque colonne textuelle pour la transformer en nombres (ex: "Voyage_Souvent" -> 2)
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    # On sauvegarde l'encodeur dans un dictionnaire pour reproduire la même transformation sur l'application web
    encoders[col] = le 

# Séparation des données : 80% pour l'entraînement de l'IA, 20% pour tester ses performances
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 4. ENTRAÎNEMENT DU MODÈLE MACHINE LEARNING
# ==========================================
print("🧠 Entraînement de l'IA (Random Forest)...")

# Initialisation du modèle Random Forest
# Le paramètre class_weight='balanced' est crucial en RH : il indique à l'IA d'accorder plus de poids 
# aux cas de "démission" (minoritaires) par rapport aux employés qui restent (majoritaires).
model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

# Apprentissage sur les données d'entraînement
model.fit(X_train, y_train)

# Évaluation de la performance sur les données de test (jamais vues par l'algorithme)
score = model.score(X_test, y_test)
print(f"🎯 Précision globale du modèle : {score * 100:.2f}%")

# ==========================================
# 5. SAUVEGARDE DES ARTEFACTS (SÉRIALISATION)
# ==========================================
print("💾 Sauvegarde des modèles...")

# Sauvegarde du "cerveau" entraîné
joblib.dump(model, 'models/hr_model.pkl')
# Sauvegarde des règles de traduction du texte en nombres
joblib.dump(encoders, 'models/encoders.pkl')
# Sauvegarde de l'ordre exact des colonnes (évite les erreurs de format dans Streamlit)
joblib.dump(list(X.columns), 'models/features.pkl')

print("✅ Succès ! Les fichiers .pkl sont prêts dans le dossier 'models/'.")
